In [0]:
import os
import time
import requests
import pandas as pd
from datetime import datetime, timezone, timedelta
from pyspark.sql import functions as F

# Date mode parameters
dbutils.widgets.dropdown("mode", "auto", ["auto", "manual"])
dbutils.widgets.text("start_date", "", "Manual Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "", "Manual End Date (YYYY-MM-DD)")

MODE = dbutils.widgets.get("mode")

CATALOG = "workspace"
DB = f"{CATALOG}.complaints"
WATERMARK = f"{DB}.pipeline_watermark"

# Target workspace samples folder[cite: 5, 8]
OUTPUT_DIR = "/Workspace/Users/furyboy4592@gmail.com/Consumer-Financial-Complaint-Analytics/data/samples"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_URL = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
BATCH_ID = f"batch_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"

# Extraction date range resolve karna[cite: 1]
def get_date_window():
    if MODE == "manual":
        return (datetime.strptime(dbutils.widgets.get("start_date"), "%Y-%m-%d").date(),
                datetime.strptime(dbutils.widgets.get("end_date"), "%Y-%m-%d").date())
    try:
        wm = spark.table(WATERMARK).filter("layer = 'silver'").select("last_processed_date").collect()
        start = wm[0][0] + timedelta(days=1) if (wm and wm[0][0]) else (datetime.now(timezone.utc).date() - timedelta(days=1))
    except Exception:
        start = datetime.now(timezone.utc).date() - timedelta(days=1)
    return start, datetime.now(timezone.utc).date()

# CFPB API se single page fetch karna[cite: 5, 6]
def fetch_api_page(date_min, date_max, frm, page_size=100, retries=3):
    params = {
        "size": page_size,
        "frm": frm,
        "date_received_min": date_min,
        "date_received_max": date_max,
        "sort": "created_date_desc"
    }
    for _ in range(retries):
        try:
            res = requests.get(BASE_URL, params=params, headers={"Accept": "application/json"}, timeout=30)
            res.raise_for_status()
            return res.json().get("hits", {}).get("hits", [])
        except requests.exceptions.RequestException:
            time.sleep(2)
    return []

# Bina kisi threshold ya row limit ke pura data paginate karke extract karna
def extract_all_data(start, end):
    rows = []
    curr = start
    while curr <= end:
        d1 = curr.isoformat()
        d2 = (curr + timedelta(days=1)).isoformat()
        frm = 0
        while True:
            hits = fetch_api_page(d1, d2, frm)
            if not hits:
                break
            rows.extend([h["_source"] for h in hits])
            frm += 100
            time.sleep(0.1)
        print(f"Date {d1} complete: ab tak total {len(rows)} rows extract hui")
        curr += timedelta(days=1)
    return pd.DataFrame(rows)

# Pipeline execute karna
start_dt, end_dt = get_date_window()
if start_dt > end_dt:
    dbutils.notebook.exit("UP_TO_DATE")

print(f"Bina kisi limit ke extraction start: {start_dt} se {end_dt}")
df = extract_all_data(start_dt, end_dt)

if df.empty:
    dbutils.notebook.exit("NO_DATA")

# Saara data bina cut-off ke direct CSV me save hoga[cite: 5, 8]
out_file = f"{OUTPUT_DIR}/incremental_load_sample.csv"
df.to_csv(out_file, index=False)
print(f"Total {len(df)} records bina kisi limit ke save ho chuke hain: {out_file}")